<a href="https://colab.research.google.com/github/devbia-ferna/Hack_RAG/blob/Hack/RAG_Hacktown_MEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Municipal Assistant with RAG**

Project-Specific Information
Project Description

Problem: Finding out which permit is required, which documents must be submitted, or how much a municipal service costs often requires visiting the municipal office in person. The information exists, but it is scattered across different sources and written in complex administrative language.

Solution: A RAG-based assistant that answers real questions about municipal services in plain language and with visible source references. If the information does not exist in the available data, the assistant explicitly states that it does not know and provides the Municipality's contact information instead of generating an unsupported answer.

Source Data (public): Municipality portal – https://www.cm-guimaraes.pt/servicos-online/formularios




**0. Installation**

In [ ]:
%pip install "langchain==0.3.27" "langchain-community==0.3.31" "langchain-chroma==0.2.6" "langchain-core>=0.3.0,<0.4.0" "langchain-huggingface>=0.3,<1" "langchain-google-genai>=2.1,<3" "langchain-openai" sentence-transformers pypdf beautifulsoup4 requests gradio reportlab -qqq

%pip install gTTS faster-whisper -qqq

%pip install --upgrade langchain-huggingface sentence-transformers huggingface_hub


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.8/407.8 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 7.

**1. Setup**

In [ ]:
import os, re, json, time
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
import gradio as gr
from google.colab import userdata

from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
# ==================================================
# API keys — M.E.L. (DeepSeek)
# ==================================================
# No Google Colab, adiciona DEEPSEEK_API_KEY em Secrets (ícone da chave).
# Não escrevas nem publiques a chave diretamente no notebook.
from google.colab import userdata

DEEPSEEK_API_KEY = userdata.get("DEEPSEEK_API_KEY")
if not DEEPSEEK_API_KEY:
    raise ValueError(
        "Falta configurar DEEPSEEK_API_KEY em Colab > Secrets. "
        "Ativa o acesso ao notebook e volta a executar esta célula."
    )

os.environ["DEEPSEEK_API_KEY"] = DEEPSEEK_API_KEY.strip()
print("Chave do M.E.L. carregada com segurança.")


Chave do M.E.L. carregada com segurança.


In [ ]:
BASE_URL = "https://www.cm-guimaraes.pt"
LISTING_URL = f"{BASE_URL}/servicos-online/formularios"
PROC_MARKER = "detalhe-de-procedimento"   # só os links com isto são procedimentos reais

HEADERS = {"User-Agent": "Mozilla/5.0 (projeto Hack4Guimaraes 2026 - assistente RAG)"}

MAX_PROCEDURES = 60        # nº de procedimentos a ingerir (None = todos)
REQUEST_DELAY = 0.5
CACHE_FILE = "procedimentos_guimaraes.json"
CHROMA_DIR = "./chroma_guimaraes"
COLLECTION = "m.e.l_rag_db"

# Score mínimo de relevância (0 a 1, distância cosseno). Se responder "não sei" demasiado, baixa; se responder fora do tema, sobe.
MIN_SCORE = 0.20

# O modelo local (SmolLM2) ficou no Anexo, desligado por defeito. Põe True se quiseres carregá-lo (precisa de GPU).
USAR_MODELO_LOCAL = False


INFO_GERAL = """Informação geral sobre os formulários da Câmara Municipal de Guimarães.
Nos Serviços Online é possível preencher online os formulários, divididos por áreas.
O preenchimento online só é possível se o munícipe estiver registado e autenticado.
Dúvidas no preenchimento dos formulários: contactar o serviço responsável.
Telefone: 253 421 200 (chamada para a rede fixa nacional). E-mail: geral@cm-guimaraes.pt.
Morada: Largo Cónego José Maria Gomes, 4804-534 Guimarães."""

# Linhas de menu/cookies que aparecem em todas as páginas e só poluem a pesquisa
BOILERPLATE = [
    "Passar para o Conteúdo Principal", "Utilizamos cookies", "Personalizar cookies",
    "Aceitar todos os Cookies", "Voltar ao topo da página",
]

# Layer 1 - Ingestion
(syntax reviewed by an AI agent, created using Copilot Agent Builder)

### Fase 1: Ingestão de Dados e Construção do Catálogo

Nesta fase inicial, o assistente acede ao portal de formulários da Câmara Municipal de Guimarães para mapear os procedimentos e serviços disponíveis. Para evitar sobrecarregar o servidor municipal, implementamos limites e pequenos atrasos entre cada pedido (`REQUEST_DELAY`).

In [ ]:
# ==================================================
# Folder Structure
# ==================================================

DATA_DIR = "data"

RAW_HTML_DIR = f"{DATA_DIR}/raw/html"
RAW_PDF_DIR = f"{DATA_DIR}/raw/pdfs"

PROCESSED_DIR = f"{DATA_DIR}/processed"

os.makedirs(RAW_HTML_DIR, exist_ok=True)
os.makedirs(RAW_PDF_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Structure created.")

Structure created.


In [ ]:
# ==================================================
# Utilities
# ==================================================

def clean_text(text):
    if not text:
        return ""

    text = re.sub(r"[ \t\r\f\v]+", " ", text)
    linhas = [l.strip() for l in text.splitlines()]
    linhas = [l for l in linhas if l and not any(b in l for b in BOILERPLATE)]

    return "\n".join(linhas).strip()


def absolute_url(link):
    return urljoin(BASE_URL, link)

In [ ]:
_HTML_CACHE = {}   # evita pedir a mesma página duas vezes (texto + PDFs)

def fetch_html(url, tentativas=3):
    """Descarrega o HTML (UTF-8). Tenta de novo se o site demorar ou recusar um pedido."""
    if url in _HTML_CACHE:
        return _HTML_CACHE[url]
    for t in range(1, tentativas + 1):
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            r.raise_for_status()
            if r.encoding == 'ISO-8859-1':
                r.encoding = 'utf-8'
            _HTML_CACHE[url] = r.text
            return r.text
        except Exception as e:
            print(f"ERRO ao aceder a {url} (tentativa {t}/{tentativas}): {type(e).__name__}: {str(e)[:120]}")
            if t < tentativas:
                time.sleep(3 * t)
    return None

In [ ]:
def discover_services():
    """Lê a listagem e devolve os procedimentos (com a área a que pertencem).
    1.º tenta só os links de procedimentos; se não houver, usa o filtro de links da versão anterior."""
    html = fetch_html(LISTING_URL)
    if not html:
        return []

    soup = BeautifulSoup(html, "html.parser")
    services = {}

    # --- Caminho principal: só links de procedimentos
    for a in soup.find_all("a", href=True):
        if PROC_MARKER not in a["href"]:
            continue
        title = a.get_text(" ", strip=True)
        if len(title) < 4:
            continue
        if title.isupper():
            title = title.capitalize()

        h = a.find_previous(["h2", "h3", "h4", "h5"])
        area = h.get_text(" ", strip=True) if h else ""
        url = absolute_url(a["href"])
        services[url] = {"title": title, "url": url, "area": area}

    if services:
        services = list(services.values())
        print(f"{len(services)} procedimentos encontrados")
        return services

    # --- Plano B: filtro de links institucionais (da versão anterior do notebook)
    print("AVISO: não encontrei links de procedimentos. A usar o filtro de links institucionais.")
    ignorar_padroes = [
        "accessibility", "cookie", "topo-da-pagina", "sticky", "heraldica",
        "camara-municipal", "assembleia-municipal", "freguesias", "eleicoes",
        "facebook", "twitter", "instagram", "youtube", "linkedin"
    ]
    for a in soup.find_all("a", href=True):
        href = a["href"]
        title = a.get_text(" ", strip=True)
        if len(title) < 4:
            continue
        if href.startswith("#") or any(p in href.lower() or p in title.lower() for p in ignorar_padroes):
            continue
        url = absolute_url(href)
        services[url] = {"title": title, "url": url, "area": ""}

    services = list(services.values())
    print(f"{len(services)} serviços de formulários válidos encontrados após filtragem.")
    return services

In [ ]:
services = discover_services()

services[:10]

54 procedimentos encontrados


[{'title': 'Atendimento',
  'url': 'https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI0FTQ0VOU09SI0FTQ0VO',
  'area': 'ascensores'},
 {'title': 'Atendimento',
  'url': 'https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI0FTQ0VOU09SI0FTQ0VOU09S',
  'area': 'ATENDIMENTO'},
 {'title': 'Atendimento',
  'url': 'https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI0NBLURFRjEjQ0EtREVGMQ==',
  'area': 'candidatura a apoio à pessoa com deficiência'},
 {'title': 'Atendimento',
  'url': 'https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI0NBLUlETyNDQS1JRE8=',
  'area': 'cartão municipal sénior'},
 {'title': 'Atendimento',
  'url': 'https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI0NFTSNDRU0wMQ==',
  'area': 'cemitérios municipais'},
 {'title': 'Atendimento',
  'url

In [ ]:
# ==================================================
# Download HTML
# ==================================================

def extract_page_content(url):
    html = fetch_html(url)
    if not html:
        return ""

    soup = BeautifulSoup(html, "html.parser")

    # Remover elementos sem informação útil
    for tag in soup([
        "script", "style", "noscript",
        "header", "footer", "nav", "aside"
    ]):
        tag.decompose()

    # Encontrar o conteúdo principal
    scope = (
        soup.find("main")
        or soup.find("article")
        or soup.find("body")
        or soup
    )

    # Extrair texto em linhas separadas
    text = scope.get_text("\n", strip=True)
    text = clean_text(text)

    # Guardar o título real do procedimento no conteúdo
    h1 = soup.find("h1")
    title = (
        h1.get_text(" ", strip=True)
        if h1 else ""
    )

    if title and not text.startswith(title):
        text = f"Título: {title}\n{text}"

    return text


In [ ]:
# ==================================================
# Detect PDFs
# ==================================================

def detect_pdfs(url):

    html = fetch_html(url)

    if not html:
        return []

    soup = BeautifulSoup(html, "html.parser")

    pdfs = []

    for a in soup.find_all("a", href=True):

        href = a["href"]

        if ".pdf" in href.lower():

            pdfs.append({
                "title": a.get_text(" ", strip=True),
                "url": absolute_url(href)
            })

    return pdfs

In [ ]:
import unicodedata

TITULOS_GENERICOS = {"atendimento", "formularios", "formulario", "servicos online", "detalhe de procedimento"}


def _sem_acentos(t):
    t = unicodedata.normalize("NFKD", str(t).lower())
    return "".join(c for c in t if not unicodedata.combining(c)).strip(" -:")


def _bonito(t):
    t = t.strip()
    return t.capitalize() if t.isupper() else t


def titulo_real(service, text):
    """Se o título do link for genérico ("Atendimento"), usa a 1.ª linha útil da página (ou a área)."""
    if _sem_acentos(service["title"]) not in TITULOS_GENERICOS:
        return service["title"]
    for linha in text.splitlines():
        linha = re.sub(r"^t[ií]tulo:\s*", "", linha.strip(), flags=re.I)
        if 4 <= len(linha) <= 120 and _sem_acentos(linha) not in TITULOS_GENERICOS:
            return _bonito(linha)
    area = service.get("area", "")
    if area and _sem_acentos(area) not in TITULOS_GENERICOS:
        return _bonito(area)
    return service["title"]


catalog = []
services = discover_services()

alvo = services if MAX_PROCEDURES is None else services[:MAX_PROCEDURES]

for i, service in enumerate(alvo):
    text = extract_page_content(service["url"])
    titulo = titulo_real(service, text)
    area = service.get("area", "")
    if _sem_acentos(area) in TITULOS_GENERICOS or _sem_acentos(area) == _sem_acentos(titulo):
        area = ""                      # a área só é útil quando acrescenta informação
    print(f"[{i+1}/{len(alvo)}] A recolher: {titulo}")
    pdfs = detect_pdfs(service["url"])

    catalog.append({
        "id": f"SERV_{i+1:03}",
        "title": titulo,
        "area": area,
        "url": service["url"],
        "text": text,
        "pdfs": pdfs,
        "retrieved": time.strftime("%Y-%m-%d")
    })
    time.sleep(REQUEST_DELAY)

if not catalog:
    raise RuntimeError("Não consegui obter nenhum procedimento do portal (o site pode estar lento ou ter recusado o pedido). "
                       "Espera 1 minuto e corre esta célula outra vez. Se persistir, vê a mensagem ERRO acima.")

print(f"Catálogo atualizado com {len(catalog)} serviços enriquecidos.")
print("Exemplos de títulos:", [c["title"] for c in catalog[:5]])

54 procedimentos encontrados
[1/54] A recolher: Primeira inspeção de ascensores
[2/54] A recolher: Inspeção ou reinspeção de ascensores
[3/54] A recolher: Candidatura a apoio à pessoa com deficiência
[4/54] A recolher: Cartão municipal sénior - online
[5/54] A recolher: Trabalhos a executar
[6/54] A recolher: Controlo metrológico
[7/54] A recolher: Aluguer espaços cowork
[8/54] A recolher: Declaração da pessoa com reforma antecipada
[9/54] A recolher: Aluguer pavilhões - clubes e associações
[10/54] A recolher: Residente - alteração de matrícula veículo
[11/54] A recolher: Veiculo 100% elétrico
[12/54] A recolher: RESIDENTE DENTRO CENTRO HISTÓRICO - 1º VEÍCULO
[13/54] A recolher: RESIDENTE FORA CENTRO HISTÓRICO - 1º VEÍCULO
[14/54] A recolher: RESIDENTE FORA CENTRO HISTÓRICO - 2º VEÍCULO
[15/54] A recolher: Instalação de circos
[16/54] A recolher: Festividades
[17/54] A recolher: Lançamento de fogo
[18/54] A recolher: Passeios ou caminhadas
[19/54] A recolher: Autorização especial de h

In [ ]:
# ==================================================
# Guardar o catálogo (com cópia de segurança do último catálogo bom)
# ==================================================
import shutil

CATALOG_FILE = f"{PROCESSED_DIR}/services_catalog.json"
BACKUP_FILE = f"{PROCESSED_DIR}/services_catalog_ULTIMO_BOM.json"

if not catalog:
    raise RuntimeError("O catálogo está vazio: não guardei nada para não apagar o último catálogo bom. "
                       "Volta a correr a célula anterior (catalog = []).")

with open(CATALOG_FILE, "w", encoding="utf-8") as f:
    json.dump(catalog, f, ensure_ascii=False, indent=2)
shutil.copyfile(CATALOG_FILE, BACKUP_FILE)

print(f"Guardados {len(catalog)} procedimentos em {CATALOG_FILE}")
print("Dica: descarrega o ficheiro", BACKUP_FILE, "(painel de ficheiros à esquerda) como cópia de segurança para a demo.")

Guardados 54 procedimentos em data/processed/services_catalog.json
Dica: descarrega o ficheiro data/processed/services_catalog_ULTIMO_BOM.json (painel de ficheiros à esquerda) como cópia de segurança para a demo.


In [ ]:
# ==================================================
# Ingestion Statistics
# ==================================================

total_chars = sum(
len(item["text"] or "")
for item in catalog
)

total_pdfs = sum(
len(item["pdfs"])
for item in catalog
)

print("Serviços:", len(catalog))
print("PDFs:", total_pdfs)
print("Caracteres:", total_chars)

Serviços: 54
PDFs: 22
Caracteres: 57226


**Data Ingestion**

In [ ]:
import pandas as pd

def validar_ingestao(catalogo):
    obrigatorios = {"title", "url", "text", "pdfs"}
    erros = []
    for i, item in enumerate(catalogo):
        faltam = obrigatorios - set(item.keys())
        if faltam:
            erros.append({"registo": i, "erro": f"Campos em falta: {', '.join(sorted(faltam))}"})
        if not str(item.get("url", "")).startswith("http"):
            erros.append({"registo": i, "erro": "URL em falta ou inválido"})
        if not str(item.get("title", "")).strip():
            erros.append({"registo": i, "erro": "Título vazio"})
        if not str(item.get("text", "")).strip() and not item.get("pdfs"):
            erros.append({"registo": i, "erro": "Sem texto extraído e sem PDFs detetados"})
    urls = [str(x.get("url", "")).strip() for x in catalogo if x.get("url")]
    duplicados = sorted({u for u in urls if urls.count(u) > 1})
    resumo = {"total_registos": len(catalogo), "URLs_duplicados": len(duplicados),
              "registos_com_erro": len({e['registo'] for e in erros}),
              "total_erros": len(erros)}
    return resumo, pd.DataFrame(erros), duplicados

resumo_ingestao, erros_ingestao, urls_duplicados = validar_ingestao(catalog)
print("Resumo da ingestão:", resumo_ingestao)
display(erros_ingestao.head(30))
if urls_duplicados:
    print("URLs duplicados (primeiros 20):", urls_duplicados[:20])

Resumo da ingestão: {'total_registos': 54, 'URLs_duplicados': 0, 'registos_com_erro': 0, 'total_erros': 0}


""


In [ ]:
def validar_tabela(df, nome, colunas_obrigatorias=None, chave=None):
    colunas_obrigatorias = colunas_obrigatorias or []
    problemas = []
    faltam = sorted(set(colunas_obrigatorias) - set(df.columns))
    if faltam:
        problemas.append(f"{nome}: faltam colunas {faltam}")
    for coluna in colunas_obrigatorias:
        if coluna in df.columns:
            nulos = int(df[coluna].isna().sum())
            if nulos:
                problemas.append(f"{nome}: {nulos} valores nulos em '{coluna}'")
    if chave and chave in df.columns:
        duplicados = int(df[chave].duplicated().sum())
        if duplicados:
            problemas.append(f"{nome}: {duplicados} valores duplicados na chave '{chave}'")
    return {"tabela": nome, "linhas": len(df), "colunas": len(df.columns),
            "estado": "OK" if not problemas else "VERIFICAR", "problemas": problemas}

# Exemplo de utilização (descomenta e adapta quando as tabelas estiverem disponíveis):
 #resultado_1 = validar_tabela(tabela_1, "Tabela 1", colunas_obrigatorias=["id", "title"], chave="id")
 #resultado_2 = validar_tabela(tabela_2, "Tabela 2", colunas_obrigatorias=["id", "url"], chave="id")
 #print(f"{resultado_1}\n# {resultado_2}")


In [ ]:
documents = [
    Document(page_content=item["text"],
             metadata={"source": item["url"], "title": item["title"], "area": item.get("area", "")})
    for item in catalog if item["text"]
]

# informação geral e contactos da Câmara também fazem parte da base de conhecimento
documents.append(Document(
    page_content=INFO_GERAL,
    metadata={"source": LISTING_URL, "title": "Informação geral e contactos", "area": ""},
))

text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)

# Cada chunk começa com o nome do serviço, para a pesquisa e o LLM saberem de que procedimento se trata
for c in chunks:
    cab = f"Serviço: {c.metadata['title']}"
    if c.metadata.get("area"):
        cab += f" (área: {c.metadata['area']})"
    c.page_content = cab + "\n" + c.page_content

print(f"{len(chunks)} chunks gerados a partir de {len(documents)} documentos")

96 chunks gerados a partir de 55 documentos


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# 1. Modelo de embeddings multilingue (percebe bem português; o all-MiniLM-L6-v2 foi pensado para inglês)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# 2. Se voltares a correr esta célula, apaga a coleção anterior para não duplicar chunks
try:
    Chroma(collection_name=COLLECTION, embedding_function=embeddings).delete_collection()
except Exception:
    pass

# 3. Recriar a base vetorial Chroma com os chunks gerados anteriormente
#    (distância cosseno: faz com que os scores fiquem entre 0 e 1 e o MIN_SCORE faça sentido)
vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION,
    collection_metadata={"hnsw:space": "cosine"},
)

# 4. Configurar o retriever para ir buscar os 3 chunks mais relevantes
retriever = vectordb.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("Base vetorial e retriever criados localmente com sucesso e a custo zero!")

/tmp/ipykernel_4852/541720051.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Base vetorial e retriever criados localmente com sucesso e a custo zero!


In [ ]:
# ==================================================
# Pesquisa híbrida: embeddings + correspondência com o título + reescrita da pergunta (quando nenhum título coincide com a pergunta)
# (o cidadão diz "esplanada", o formulário chama-se "Ocupação de espaço público")
# ==================================================
import unicodedata

PROMPT_REESCRITA = """Uma pessoa fez uma pergunta à Câmara Municipal de Guimarães, com palavras do dia a dia.
Escreve de 2 a 3 pesquisas curtas (2 a 5 palavras cada) com os termos administrativos que o nome do formulário ou procedimento provavelmente usa.
Exemplos: "obras na minha casa" -> "licença de obras"; "lixo grande na rua" -> "recolha de monstros".
Responde APENAS com uma lista JSON de strings.

Pergunta: __P__"""


_PARAGENS = {"como", "para", "preciso", "posso", "quero", "tenho", "algum", "alguma", "qual", "quais",
             "quando", "onde", "pedir", "peco", "fazer", "pode", "meu", "minha", "esta", "este", "isso",
             "sobre", "mais", "muito", "ainda", "vou", "ter", "uma", "umas", "uns", "dos", "das", "nos", "nas"}


def _tokens(texto):
    t = unicodedata.normalize("NFKD", str(texto).lower())
    t = "".join(c for c in t if not unicodedata.combining(c))
    return [p for p in re.findall(r"[a-z0-9]+", t) if len(p) >= 4 and p not in _PARAGENS]


class PesquisaHibrida:
    """Mesma interface do Chroma (similarity_search_with_relevance_scores), mas junta os procedimentos
    cujo título tem correspondência com a pergunta. Funciona com o resto do notebook sem mudar nada."""

    def __init__(self, vetorial, todos_os_chunks, min_fracao=0.6, reescrever=True):
        self._vetorial = vetorial
        self.min_fracao = min_fracao
        self.reescrever = reescrever
        self._cache_variantes = {}
        self.ultimo_erro = ""
        self._por_titulo = {}
        for c in todos_os_chunks:
            titulo = c.metadata.get("title", "")
            if titulo not in self._por_titulo:
                self._por_titulo[titulo] = {"chunks": [], "tokens": set(p[:5] for p in _tokens(titulo))}
            self._por_titulo[titulo]["chunks"].append(c)

    def _pesquisar(self, pergunta, k=4):
        melhores = {}
        for doc, score in self._vetorial.similarity_search_with_relevance_scores(pergunta, k=k):
            melhores[(doc.metadata.get("source"), doc.page_content)] = (doc, score)

        pergunta_tokens = set(p[:5] for p in _tokens(pergunta))
        houve_titulo = False
        for titulo, info in self._por_titulo.items():
            tt = info["tokens"]
            if not tt or titulo.strip().lower() == "atendimento":
                continue
            fracao = len(tt & pergunta_tokens) / len(tt)
            if fracao >= self.min_fracao:
                houve_titulo = True
                score = 0.55 + 0.30 * fracao                     # acima dos resultados só por embeddings
                for doc in info["chunks"][:2]:                   # o início da página é o que mais interessa
                    chave = (doc.metadata.get("source"), doc.page_content)
                    if chave not in melhores or score > melhores[chave][1]:
                        melhores[chave] = (doc, score)

        return list(melhores.values()), houve_titulo

    def _variantes(self, pergunta):
        """Pesquisas alternativas geradas pelo modelo (se falhar, devolve [] e a pesquisa normal continua)."""
        if pergunta in self._cache_variantes:
            return self._cache_variantes[pergunta]
        try:
            lista = parse_json(chamar_llm(PROMPT_REESCRITA.replace("__P__", pergunta), tentativas=1))
            variantes = [str(x).strip() for x in lista if str(x).strip()][:3]
        except Exception as e:
            self.ultimo_erro = f"{type(e).__name__}: {str(e)[:200]}"
            return []
        self._cache_variantes[pergunta] = variantes
        return variantes

    def similarity_search_with_relevance_scores(self, pergunta, k=4):
        resultados, houve_titulo = self._pesquisar(pergunta, k)
        melhores = {(d.metadata.get("source"), d.page_content): (d, sc) for d, sc in resultados}
        if self.reescrever and not houve_titulo:
            for variante in self._variantes(pergunta):
                for doc, sc in self._pesquisar(variante, k)[0]:
                    chave = (doc.metadata.get("source"), doc.page_content)
                    if chave not in melhores or sc > melhores[chave][1]:
                        melhores[chave] = (doc, sc)
        return sorted(melhores.values(), key=lambda x: x[1], reverse=True)[:max(k, 4)]

    def __getattr__(self, nome):                                  # tudo o resto vai para o Chroma
        return getattr(self._vetorial, nome)


vectordb = PesquisaHibrida(getattr(vectordb, "_vetorial", vectordb), chunks)    # seguro se correres a célula 2 vezes
print("✅ Pesquisa híbrida ativa (embeddings + títulos + reescrita da pergunta).")

✅ Pesquisa híbrida ativa (embeddings + títulos + reescrita da pergunta).


Inference Phase


### Fase 2: Configuração do Modelo de Linguagem (M.E.L. com DeepSeek)

O modelo de linguagem é o **DeepSeek Flash**, através da API (paga, mas muito barata). Os embeddings continuam a correr localmente e de forma gratuita. O modelo local SmolLM2 ficou no Anexo, desligado por defeito.

In [ ]:
NAO_SEI = ("Não encontrei essa informação nos serviços e formulários do Município que tenho disponíveis. "
           "Para não lhe dar uma resposta errada, sugiro que contacte a Câmara Municipal de Guimarães: "
           "telefone 253 421 200 ou e-mail geral@cm-guimaraes.pt.")

print("Mensagem padrão definida.")

Mensagem padrão definida.


In [ ]:
# ==================================================
# M.E.L. — LLM + PROMPT + CHAIN (DeepSeek)
# ==================================================

import os, time
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

DEEPSEEK_API_KEY = os.environ.get("DEEPSEEK_API_KEY")
if not DEEPSEEK_API_KEY:
    raise ValueError("Configura DEEPSEEK_API_KEY em Colab > Secrets e executa a célula de setup.")

# A API da DeepSeek é compatível com o formato OpenAI.
# "deepseek-flash" é o identificador atual; "deepseek-chat" fica como alternativa.
# O modo "thinking" vem ligado por defeito: desligamo-lo para as respostas serem mais rápidas e baratas.
MODELOS_DEEPSEEK = ["deepseek-flash", "deepseek-chat"]

llm = None
for MODELO in MODELOS_DEEPSEEK:
    try:
        _candidato = ChatOpenAI(
            model=MODELO,
            api_key=DEEPSEEK_API_KEY,
            base_url="https://api.deepseek.com",
            temperature=0.2,
            max_retries=2,
            timeout=60,
            extra_body={"thinking": {"type": "disabled"}},
        )
        _candidato.invoke("Diz apenas: OK")
        llm = _candidato
        break
    except Exception as e:
        print(f"  {MODELO}: não funcionou ({type(e).__name__}: {str(e)[:150]})")

if llm is None:
    raise RuntimeError("Nenhum modelo da DeepSeek respondeu. Verifica a chave e o saldo da conta.")

prompt = ChatPromptTemplate.from_template("""
És o M.E.L., o assistente ("Falar com a Câmara sem burocracia"),
para os serviços municipais de Guimarães.

Regras:
1. Responde APENAS com base no contexto fornecido. Não inventes valores, prazos, documentos nem leis.
2. Usa linguagem simples, frases curtas e português de Portugal.
3. Quando a pergunta for "como faço...", responde em passos numerados.
4. Se o contexto não contiver a resposta, diz claramente que não sabes e direciona para o contacto geral da
   Câmara Municipal de Guimarães: telefone 253 421 200 ou e-mail geral@cm-guimaraes.pt.
5. "M.E.L" é apenas o nome deste assistente. Nunca o indiques como serviço ou contacto da Câmara.

Contexto:
{context}

Pergunta:
{question}

Resposta:
""")

chain = prompt | llm | StrOutputParser()
print(f"✅ M.E.L. configurado com o modelo {MODELO}.")

✅ M.E.L. configurado com o modelo deepseek-flash.


In [ ]:
# Teste rápido do M.E.L.: se isto falhar, o erro aparece aqui em vez de ficar escondido mais à frente
try:
    response = llm.invoke("Diz apenas: Teste")
    print("M.E.L. OK:", response.content)
except Exception as e:
    print("ERRO no M.E.L.:", type(e).__name__)
    print(str(e)[:500])

M.E.L. OK: Teste


 **Pipeline RAG**

### Fase 3: Pipeline RAG (Retrieval-Augmented Generation)

Aqui unimos a base de dados vetorial ao nosso modelo local. Quando um munícipe faz uma pergunta, o ChromaDB pesquisa as fontes mais parecidas. Se o nível de semelhança for baixo (abaixo de `MIN_SCORE`), o assistente devolve uma mensagem padrão amigável sem inventar respostas.

In [ ]:
def formatar_resposta(resultado):
    """Formata o dicionário retornado pelo RAG numa string elegante para o Gradio."""
    texto = resultado["answer"]
    if resultado["sources"]:
        texto += "\n\n**Fontes:**\n" + "\n".join(f"- [{t}]({u})" for t, u in resultado["sources"])
    return texto

def rag_answer(question):
    """Devolve {"answer": texto, "sources": [(título, url), ...]}.
    Consome o M.E.L. (DeepSeek) de forma otimizada para respostas rápidas e econômicas.
    """
    # k=3 para diminuir o tamanho do contexto e poupar tokens
    resultados = vectordb.similarity_search_with_relevance_scores(question, k=3)
    relevantes = [(doc, score) for doc, score in resultados if score >= MIN_SCORE]

    if not relevantes:
        return {"answer": NAO_SEI, "sources": []}

    context = "\n---\n".join(doc.page_content for doc, _ in relevantes)

    try:
        # Se o plano gratuito atingir o limite de pedidos (429), espera e tenta de novo
        for tentativa in range(3):
            try:
                answer = chain.invoke({"context": context, "question": question})
                break
            except Exception as e:
                if tentativa < 2 and any(x in str(e).lower() for x in ("429", "quota", "resource")):
                    time.sleep(20 * (tentativa + 1))
                    continue
                raise
    except Exception as e:
        answer = f"Ocorreu um erro ao comunicar com o M.E.L. (DeepSeek) ({type(e).__name__}: {str(e)[:150]})"

    fontes, vistos = [], set()
    for doc, _ in relevantes:
        if doc.metadata["source"] not in vistos:
            vistos.add(doc.metadata["source"])
            fontes.append((doc.metadata["title"], doc.metadata["source"]))

    return {"answer": answer, "sources": fontes}

In [ ]:
# Guarda as respostas já dadas: perguntas repetidas (ensaios da demo) não gastam pedidos ao modelo
if "_rag_original" not in globals():
    _rag_original = rag_answer
_cache = {}

def rag_answer(q):
    k = q.strip().lower()
    if k not in _cache:
        r = _rag_original(q)
        if r["answer"].startswith("Ocorreu um erro"):
            return r            # não guardar erros
        _cache[k] = r
    return _cache[k]

# **Demo**

In [ ]:
pergunta_teste = "Que documentos preciso para o dístico de residente no centro histórico?"

# Ver os scores de relevância no ChromaDB (útil para ajustar o MIN_SCORE)
for doc, score in vectordb.similarity_search_with_relevance_scores(pergunta_teste, k=4):
    print(f"{score:.2f} | {doc.metadata['title']}")

print("\n--- Resposta ---")
print(formatar_resposta(rag_answer(pergunta_teste)))

0.73 | RESIDENTE FORA CENTRO HISTÓRICO - 2º VEÍCULO
0.73 | RESIDENTE FORA CENTRO HISTÓRICO - 1º VEÍCULO
0.73 | RESIDENTE DENTRO CENTRO HISTÓRICO - 1º VEÍCULO
0.73 | RESIDENTE DENTRO CENTRO HISTÓRICO - 1º VEÍCULO

--- Resposta ---
Para o dístico de residente no centro histórico, o contexto indica que precisas de:

1. **Documento comprovativo do domicílio fiscal** (obrigatório). Podes obtê-lo no Portal das Finanças: Finanças > Serviços > Documentos e Certidões > Pedir Certidão > Domicílio Fiscal > Confirmar > Obter.
2. **Certificado de matrícula ou título do registo de propriedade do veículo**, caso o veículo se encontre em outro regime de titularidade (frente e verso).
3. **Outros documentos** (opcional, não obrigatório, formato PDF).

Nota: o contexto não especifica se estes documentos se aplicam igualmente ao 1.º e ao 2.º veículo, nem detalha todos os documentos possíveis. Se tiveres dúvidas, contacta a Câmara Municipal de Guimarães: telefone 253 421 200 ou e-mail geral@cm-guimaraes.p

In [ ]:
import unicodedata
import pandas as pd

def norm(t):
    """minúsculas e sem acentos, para comparar textos"""
    t = unicodedata.normalize("NFKD", str(t).lower())
    return "".join(c for c in t if not unicodedata.combining(c))

# tipo: "facto" (fonte + dado na resposta), "fonte" (só a fonte certa), "fora" (deve dizer que não sabe)
CASOS = [
    # factos confirmados na página do Cartão Municipal Sénior
    {"p": "Quanto custa a segunda via do cartão municipal sénior?",          "tipo": "facto", "fonte": "nior", "facto": ["1,00"]},
    {"p": "Que idade é preciso ter para pedir o cartão municipal sénior?",   "tipo": "facto", "fonte": "nior", "facto": ["65"]},
    {"p": "Que documentos preciso para pedir o cartão municipal sénior?",    "tipo": "facto", "fonte": "nior", "facto": ["residência", "residencia", "eleitor", "fotografia"]},
    {"p": "Que vantagens tem o cartão municipal sénior?",                    "tipo": "facto", "fonte": "nior", "facto": ["teleférico", "teleferico", "piscinas", "40%"]},
    {"p": "Posso pedir o cartão municipal sénior online?",                   "tipo": "facto", "fonte": "nior", "facto": ["online", "autentic"]},
    # a pesquisa encontra o procedimento certo?
    {"p": "Como peço a inspeção de um elevador?",                            "tipo": "fonte", "fonte": "ascensor"},
    {"p": "Preciso de autorização para lançar fogo?",                        "tipo": "fonte", "fonte": "fogo"},
    {"p": "Como peço autorização para ocupar espaço público?",               "tipo": "fonte", "fonte": "espa"},
    {"p": "Como altero a matrícula do veículo de residente?",                "tipo": "fonte", "fonte": "matr"},
    {"p": "Como posso alugar um espaço de cowork?",                          "tipo": "fonte", "fonte": "cowork"},
    {"p": "Como peço apoio para uma pessoa com deficiência?",                "tipo": "fonte", "fonte": "defici"},
    {"p": "Como funciona o controlo metrológico?",                           "tipo": "fonte", "fonte": "metrol"},
    {"p": "O meu clube pode alugar um pavilhão municipal?",                  "tipo": "fonte", "fonte": "pavilh"},
    {"p": "Como instalo um circo no concelho?",                              "tipo": "fonte", "fonte": "circo"},
    {"p": "Como peço um horário especial de funcionamento para o meu café?", "tipo": "fonte", "fonte": "hor"},
    {"p": "Tenho um veículo elétrico, há alguma vantagem no estacionamento?", "tipo": "fonte", "fonte": "eletric"},
    {"p": "Como peço a declaração de reforma antecipada?",                   "tipo": "fonte", "fonte": "reforma"},
    # fora do âmbito: tem de dizer que não sabe
    {"p": "Qual é a capital de França?",                                     "tipo": "fora"},
    {"p": "Quem ganhou o campeonato de futebol este ano?",                   "tipo": "fora"},
    {"p": "Qual é a receita de bacalhau à Gomes de Sá?",                     "tipo": "fora"},
]

PAUSA = 6   # segundos entre perguntas, para não bater no limite do plano gratuito do Gemini

In [ ]:
def rag_sem_llm(question):
    """Só a pesquisa (não gasta quota do Gemini): devolve as fontes que o assistente usaria."""
    res = vectordb.similarity_search_with_relevance_scores(question, k=3)
    rel = [(d, s) for d, s in res if s >= MIN_SCORE]
    fontes, vistos = [], set()
    for d, _ in rel:
        if d.metadata["source"] not in vistos:
            vistos.add(d.metadata["source"])
            fontes.append((d.metadata["title"], d.metadata["source"]))
    return {"answer": "" if rel else NAO_SEI, "sources": fontes}

In [ ]:
def avaliar(casos, pausa=PAUSA):
    linhas = []
    for i, c in enumerate(casos, 1):
        try:
            r = (rag_sem_llm if c["tipo"] == "fonte" else rag_answer)(c["p"])
            resposta, fontes = r["answer"], r["sources"]
            erro = ""
        except Exception as e:
            resposta, fontes, erro = "", [], f"{type(e).__name__}: {str(e)[:80]}"

        titulos = " | ".join(t for t, _ in fontes)
        fonte_ok = facto_ok = None
        if c["tipo"] == "fora":
            ok = (not erro) and ("253 421 200" in resposta)   # só o "não sei" traz o contacto da Câmara
        else:
            fonte_ok = norm(c["fonte"]) in norm(titulos)
            if c["tipo"] == "facto":
                facto_ok = any(norm(f) in norm(resposta) for f in c["facto"])
            ok = (not erro) and fonte_ok and (facto_ok is not False)

        linhas.append({"n": i, "tipo": c["tipo"], "pergunta": c["p"], "ok": ok,
                       "fonte_certa": fonte_ok, "facto_certo": facto_ok,
                       "fontes": titulos, "resposta": resposta, "erro": erro})
        print(f"[{i:02}/{len(casos)}] {'OK ' if ok else 'FALHA'} {c['p']}")
        time.sleep(pausa)
    return pd.DataFrame(linhas)


resultados = avaliar(CASOS)
resultados.to_csv("avaliacao.csv", index=False, encoding="utf-8-sig")

[01/20] OK  Quanto custa a segunda via do cartão municipal sénior?
[02/20] OK  Que idade é preciso ter para pedir o cartão municipal sénior?
[03/20] OK  Que documentos preciso para pedir o cartão municipal sénior?
[04/20] OK  Que vantagens tem o cartão municipal sénior?
[05/20] OK  Posso pedir o cartão municipal sénior online?
[06/20] OK  Como peço a inspeção de um elevador?
[07/20] OK  Preciso de autorização para lançar fogo?
[08/20] OK  Como peço autorização para ocupar espaço público?
[09/20] OK  Como altero a matrícula do veículo de residente?
[10/20] OK  Como posso alugar um espaço de cowork?
[11/20] OK  Como peço apoio para uma pessoa com deficiência?
[12/20] OK  Como funciona o controlo metrológico?
[13/20] OK  O meu clube pode alugar um pavilhão municipal?
[14/20] OK  Como instalo um circo no concelho?
[15/20] OK  Como peço um horário especial de funcionamento para o meu café?
[16/20] OK  Tenho um veículo elétrico, há alguma vantagem no estacionamento?
[17/20] OK  Como peço a d

In [ ]:
def pct(s):
    s = s.dropna()
    return f"{int(s.sum())}/{len(s)} ({100 * s.mean():.0f}%)" if len(s) else "n/a"

print("RESULTADO GLOBAL:          ", pct(resultados["ok"]))
print("Encontra o procedimento:   ", pct(resultados.loc[resultados.tipo != "fora", "fonte_certa"].astype(float)))
print("Resposta com o dado certo: ", pct(resultados["facto_certo"].astype(float)))
print('"Não sei" quando devia:    ', pct(resultados.loc[resultados.tipo == "fora", "ok"]))
print("Erros técnicos:            ", int((resultados.erro != "").sum()))

print("\nCasos que falharam:")
falhas = resultados[~resultados.ok]
for _, r in falhas.iterrows():
    print(f"- [{r.n}] {r.pergunta}\n    fontes: {r.fontes or '(nenhuma)'}\n    resposta: {r.resposta[:200]!r}")
if falhas.empty:
    print("(nenhum)")

RESULTADO GLOBAL:           20/20 (100%)
Encontra o procedimento:    17/17 (100%)
Resposta com o dado certo:  5/5 (100%)
"Não sei" quando devia:     3/3 (100%)
Erros técnicos:             0

Casos que falharam:
(nenhum)


In [ ]:
def parse_json(texto):
    """Extrai JSON de uma resposta do LLM, mesmo que venha dentro de ```json ... ```."""
    texto = texto.strip()
    texto = re.sub(r"^```(?:json)?\s*|\s*```$", "", texto, flags=re.IGNORECASE).strip()
    try:
        return json.loads(texto)
    except Exception:
        m = re.search(r"(\{.*\}|\[.*\])", texto, re.DOTALL)
        if m:
            return json.loads(m.group(1))
        raise


def chamar_llm(prompt_texto, tentativas=3):
     """Chama o Gemini e espera um pouco se o plano gratuito atingir o limite de pedidos."""
     for i in range(tentativas):
         try:
             conteudo = llm.invoke(prompt_texto).content
             if isinstance(conteudo, list):
                 conteudo = "".join(p if isinstance(p, str) else p.get("text", "") for p in conteudo)
             return conteudo
         except Exception as e:
             msg = str(e).lower()
             if i < tentativas - 1 and ("429" in msg or "quota" in msg or "resource" in msg):
                 time.sleep(20 * (i + 1))
                 continue
             raise

# **Prompts**

### Fase 4: Planeador de Situações Complexas

Esta funcionalidade permite ao utilizador descrever um cenário completo (ex: "Vou abrir um café") e receber um plano estruturado passo a passo, incluindo checklist de documentos necessários obtidos localmente, estimativa de custos/prazos e um botão de exportação para PDF profissional.

In [ ]:
CONSULTAS_PROMPT = """És um assistente da Câmara Municipal de Guimarães. Uma pessoa descreveu uma situação.
Indica de 2 a 4 pesquisas curtas (3 a 8 palavras cada) para encontrar, no portal de formulários da Câmara,
os procedimentos que essa situação pode exigir.
Responde APENAS com uma lista JSON de strings.

Situação: __SITUACAO__"""


PLANO_PROMPT = """És o assistente "Falar com a Câmara sem burocracia" do Município de Guimarães.
Uma pessoa descreveu a sua situação e precisa de saber o que fazer junto da Câmara.

Com base APENAS no contexto abaixo, cria um plano com os procedimentos necessários, por ordem lógica.

Regras:
1. Não inventes procedimentos, documentos, valores, prazos nem leis. Se o contexto não indicar um custo ou um prazo, escreve "Não indicado".
2. Só inclui procedimentos que o contexto mostre serem relevantes para a situação. Se o contexto não cobrir a situação, devolve "passos": [].
3. Linguagem simples, frases curtas, português de Portugal.
4. Em "fontes" indica os números das fontes usadas em cada passo (por exemplo [1, 3] para [F1] e [F3]).
5. Em "avisos" indica o que a pessoa deve confirmar com a Câmara (por exemplo, licenças que o contexto não cobre).
6. Responde APENAS com JSON válido, sem texto antes nem depois, neste formato:
{
  "resumo": "uma frase a explicar o plano",
  "passos": [
    {
      "titulo": "nome do procedimento",
      "o_que_fazer": "o que a pessoa tem de fazer, em 1 a 3 frases",
      "documentos": ["documento 1", "documento 2"],
      "custo": "valor ou Não indicado",
      "prazo": "prazo ou Não indicado",
      "fontes": [1]
    }
  ],
  "avisos": ["aviso 1"]
}

Situação: __SITUACAO__

Contexto:
__CONTEXTO__
"""

In [ ]:
def gerar_consultas(situacao):
    """O LLM propõe pesquisas; se falhar, usa a própria situação."""
    try:
        lista = parse_json(chamar_llm(CONSULTAS_PROMPT.replace("__SITUACAO__", situacao)))
        consultas = [str(c) for c in lista if str(c).strip()][:4]
    except Exception:
        consultas = []
    return [situacao] + consultas

def recolher_contexto(consultas, k=4, max_chunks=12):
    """Junta os chunks relevantes de todas as pesquisas, sem repetidos, dos mais para os menos relevantes.
    Com a distância cosseno os scores ficam entre 0 e 1, por isso já podemos filtrar com o MIN_SCORE
    (assim, se a situação não tiver nada a ver com os serviços, o assistente diz que não sabe)."""
    melhores = {}
    for q in consultas:
        for doc, score in vectordb.similarity_search_with_relevance_scores(q, k=k):
            if score < MIN_SCORE:
                continue
            chave = (doc.metadata["source"], doc.page_content)
            if chave not in melhores or score > melhores[chave][1]:
                melhores[chave] = (doc, score)
    ordenados = sorted(melhores.values(), key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in ordenados[:max_chunks]]

def normalizar_plano(plano, n_fontes):
    """Garante que o JSON do LLM tem a forma esperada (e só fontes que existem)."""
    if not isinstance(plano, dict):
        plano = {}
    passos = []
    for p in plano.get("passos") or []:
        if not isinstance(p, dict) or not str(p.get("titulo", "")).strip():
            continue
        fontes = []
        for f in p.get("fontes") or []:
            try:
                f = int(f)
            except Exception:
                continue
            if 1 <= f <= n_fontes and f not in fontes:
                fontes.append(f)
        passos.append({
            "titulo": str(p.get("titulo")).strip(),
            "o_que_fazer": str(p.get("o_que_fazer") or "").strip(),
            "documentos": [str(d).strip() for d in (p.get("documentos") or []) if str(d).strip()],
            "custo": str(p.get("custo") or "Não indicado").strip(),
            "prazo": str(p.get("prazo") or "Não indicado").strip(),
            "fontes": fontes,
        })
    return {
        "resumo": str(plano.get("resumo") or "").strip(),
        "passos": passos,
        "avisos": [str(a).strip() for a in (plano.get("avisos") or []) if str(a).strip()],
    }

def gerar_plano(situacao):
    """Devolve (plano, fontes) com fontes = [(título, url)] numeradas a partir de 1."""
    docs = recolher_contexto(gerar_consultas(situacao))
    if not docs:
        return {"resumo": "", "passos": [], "avisos": []}, []

    fontes, indice, blocos = [], {}, []
    for doc in docs:
        url = doc.metadata["source"]
        if url not in indice:
            fontes.append((doc.metadata["title"], url))
            indice[url] = len(fontes)
        blocos.append(f"[F{indice[url]}] {doc.page_content}")

    prompt_plano = (PLANO_PROMPT
                    .replace("__SITUACAO__", situacao)
                    .replace("__CONTEXTO__", "\n\n---\n\n".join(blocos)))
    plano = normalizar_plano(parse_json(chamar_llm(prompt_plano)), len(fontes))

    usadas = []
    for p in plano["passos"]:
        for f in p["fontes"]:
            if f not in usadas:
                usadas.append(f)
    if usadas:
        mapa = {antigo: novo for novo, antigo in enumerate(usadas, 1)}
        for p in plano["passos"]:
            p["fontes"] = [mapa[f] for f in p["fontes"]]
        fontes = [fontes[i - 1] for i in usadas]

    return plano, fontes

In [ ]:
CONTACTO = "Câmara Municipal de Guimarães · telefone 253 421 200 · geral@cm-guimaraes.pt"

# Garantir que as definições de prompt locais existem como fallback seguro
PLANO_PROMPT_FALLBACK = """És o assistente "Falar com a Câmara sem burocracia" do Município de Guimarães.
Uma pessoa descreveu a sua situação e precisa de saber o que fazer junto da Câmara.
Responde APENAS com JSON válido.
"""

def plano_para_markdown(plano, fontes, situacao):
    linhas = [f"## Plano para: {situacao}"]
    if plano.get("resumo"):
        linhas.append(plano["resumo"])

    for i, p in enumerate(plano.get("passos") or [], 1):
        linhas.append(f"\n### {i}. {p.get('titulo', '')}")
        if p.get("o_que_fazer"):
            linhas.append(p["o_que_fazer"])
        if p.get("documentos"):
            linhas.append("- **Documentos:** " + "; ".join(p["documentos"]))
        linhas.append(f"- **Custo:** {p.get('custo', 'Não indicado')}")
        linhas.append(f"- **Prazo:** {p.get('prazo', 'Não indicado')}")
        if p.get("fontes"):
            linhas.append("- **Fonte:** " + ", ".join(f"[{n}]({fontes[n - 1][1]})" for n in p["fontes"] if n <= len(fontes)))

    com_docs = [p for p in (plano.get("passos") or []) if p.get("documentos")]
    if com_docs:
        linhas.append("\n### ✅ Checklist de documentos")
        for p in com_docs:
            linhas.append(f"**{p.get('titulo', '')}**")
            linhas.extend(f"- [ ] {d}" for d in p.get("documentos", []))

    if plano.get("avisos"):
        linhas.append("\n### ⚠️ A confirmar com a Câmara")
        linhas.extend(f"- {a}" for a in plano["avisos"])

    if fontes:
        linhas.append("\n**Fontes:**")
        linhas.extend(f"{n}. [{t}]({u})" for n, (t, u) in enumerate(fontes, 1))

    linhas.append(f"\n*Informação gerada automaticamente a partir do portal da Câmara. Em caso de dúvida: {CONTACTO}.*")
    return "\n".join(linhas)

def plano_para_pdf(plano, fontes, situacao, caminho):
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import cm
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
    from xml.sax.saxutils import escape

    e = lambda t: escape(str(t))
    est = getSampleStyleSheet()
    corpo = est["BodyText"]
    item = ParagraphStyle("item", parent=corpo, leftIndent=16, firstLineIndent=-16)
    peq = ParagraphStyle("peq", parent=corpo, fontSize=8, textColor=colors.grey)

    doc = SimpleDocTemplate(caminho, pagesize=A4, leftMargin=2 * cm, rightMargin=2 * cm,
                            topMargin=2 * cm, bottomMargin=2 * cm,
                            title="O meu plano com a Câmara de Guimarães")
    el = [
        Paragraph("O meu plano com a Câmara de Guimarães", est["Title"]),
        Paragraph(f"<b>Situação:</b> {e(situacao)}", corpo),
        Paragraph(f"Gerado em {time.strftime('%d/%m/%Y')}", peq),
        Spacer(1, 8),
    ]
    if plano.get("resumo"):
        el.append(Paragraph(e(plano["resumo"]), corpo))

    com_docs = [p for p in (plano.get("passos") or []) if p.get("documentos")]
    if com_docs:
        el.append(Paragraph("Checklist de documentos", est["Heading2"]))
        for p in com_docs:
            el.append(Paragraph(f"<b>{e(p.get('titulo', ''))}</b>", corpo))
            for d in p.get("documentos", []):
                el.append(Paragraph(f"[&nbsp;&nbsp;&nbsp;] {e(d)}", item))
            el.append(Spacer(1, 4))

    el.append(Paragraph("Passos", est["Heading2"]))
    for i, p in enumerate(plano.get("passos") or [], 1):
        el.append(Paragraph(f"{i}. {e(p.get('titulo', ''))}", est["Heading3"]))
        if p.get("o_que_fazer"):
            el.append(Paragraph(e(p["o_que_fazer"]), corpo))
        el.append(Paragraph(f"<b>Custo:</b> {e(p.get('custo', 'Não indicado'))}", corpo))
        el.append(Paragraph(f"<b>Prazo:</b> {e(p.get('prazo', 'Não indicado'))}", corpo))
        if p.get("fontes"):
            refs = ", ".join(
                f'<a href="{e(fontes[n - 1][1])}" color="blue">{n}</a>' for n in p["fontes"] if n <= len(fontes))
            el.append(Paragraph(f"<b>Fonte:</b> {refs}", corpo))

    if plano.get("avisos"):
        el.append(Paragraph("A confirmar com a Câmara", est["Heading2"]))
        for a in plano["avisos"]:
            el.append(Paragraph(f"• {e(a)}", item))

    if fontes:
        el.append(Paragraph("Fontes", est["Heading2"]))
        for n, (t, u) in enumerate(fontes, 1):
            el.append(Paragraph(f'{n}. <a href="{e(u)}" color="blue">{e(t)}</a>', item))

    el.append(Spacer(1, 14))
    el.append(Paragraph(
        "Informação gerada automaticamente a partir do portal de formulários da Câmara. "
        f"Em caso de dúvida: {e(CONTACTO)}.", peq))

    doc.build(el)
    return caminho

def planear(situacao):
    """Função integrada para usar o planeador do M.E.L. (DeepSeek) com robustez."""
    situacao = (situacao or "").strip()
    if len(situacao) < 5:
        return "Descreva a sua situação em poucas palavras. Por exemplo: *vou abrir um café*.", None
    try:
        plano, fontes = gerar_plano(situacao)
    except Exception as ex:
        return (f"Não consegui preparar o plano agora ({type(ex).__name__}: {str(ex)[:150]}). "
                "Tente novamente dentro de um minuto."), None
    if not plano.get("passos"):
        return NAO_SEI, None
    pdf = plano_para_pdf(plano, fontes, situacao, f"plano_{int(time.time())}.pdf")
    return plano_para_markdown(plano, fontes, situacao), pdf

In [ ]:
# Instalar o ReportLab caso não esteja presente no ambiente
try:
    import reportlab
except ImportError:
    print("A instalar a biblioteca ReportLab para exportação de PDFs...")
    %pip install reportlab -qqq

from IPython.display import display, Markdown
import time

# Garantir que as variáveis críticas estão instanciadas localmente para o teste
if 'CONSULTAS_PROMPT' not in globals():
    CONSULTAS_PROMPT = """És um assistente da Câmara Municipal de Guimarães. Uma pessoa descreveu uma situação.
Indica de 2 a 4 pesquisas curtas (3 a 8 palavras cada) para encontrar, no portal de formulários da Câmara,
os procedimentos que essa situação pode exigir.
Responde APENAS com uma lista JSON de strings.

Situação: __SITUACAO__"""

if 'PLANO_PROMPT' not in globals():
    PLANO_PROMPT = """És o assistente "Falar com a Câmara sem burocracia" do Município de Guimarães.
Uma pessoa descreveu a sua situação e precisa de saber o que fazer junto da Câmara.

Com base APENAS no contexto abaixo, cria um plano com os procedimentos necessários, por ordem lógica.

Regras:
1. Não inventes procedimentos, documentos, valores, prazos nem leis. Se o contexto não indicar um custo ou um prazo, escreve "Não indicado".
2. Só inclui procedimentos que o contexto mostre serem relevantes para a situação.
3. Linguagem simples, frases curtas, português de Portugal.
4. Responde APENAS com JSON válido.

Situação: __SITUACAO__

Contexto:
__CONTEXTO__"""

# Executar o planeamento com o Gemini
texto_plano, pdf_plano = planear("Vou abrir um café no centro de Guimarães")

display(Markdown(texto_plano))
print("PDF gerado com sucesso:", pdf_plano)

## Plano para: Vou abrir um café no centro de Guimarães
Para abrir um café no centro de Guimarães, pode ser necessário pedir autorização para ocupar espaço público, colocar publicidade e, se quiser abrir fora do horário normal, pedir autorização especial de horário de funcionamento.

### 1. Ocupação de espaço público
Se o café tiver esplanada ou outro uso de espaço público, preencha o formulário online ou entregue o requerimento presencial. Junte os anexos indicados.
- **Documentos:** Planta identificativa da localização; Planta de implantação; Memória descritiva; Outros elementos
- **Custo:** 33,00 € (taxa de apreciação)
- **Prazo:** Não indicado
- **Fonte:** [1](https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI09WUC1TTyNPVlAtU08=)

### 2. Publicidade
Se for colocar publicidade no café, preencha o formulário online ou entregue o requerimento presencial. Junte os anexos indicados.
- **Documentos:** Planta identificativa da localização; Memória descritiva; Fotografia do equipamento
- **Custo:** 33,00 € (taxa de apreciação)
- **Prazo:** Não indicado
- **Fonte:** [2](https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI1BVQiNQVUI=)

### 3. Autorização especial de horário de funcionamento
Se quiser funcionar fora do horário normal, preencha o formulário online ou entregue o requerimento presencial. Junte os documentos pedidos.
- **Documentos:** Outros documentos
- **Custo:** 36,00 € (pedido excecional de horário)
- **Prazo:** Não indicado
- **Fonte:** [3](https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI0hPUi5FWEMjSE9SLkVYQw==)

### ✅ Checklist de documentos
**Ocupação de espaço público**
- [ ] Planta identificativa da localização
- [ ] Planta de implantação
- [ ] Memória descritiva
- [ ] Outros elementos
**Publicidade**
- [ ] Planta identificativa da localização
- [ ] Memória descritiva
- [ ] Fotografia do equipamento
**Autorização especial de horário de funcionamento**
- [ ] Outros documentos

### ⚠️ A confirmar com a Câmara
- Confirme com a Câmara se precisa de licença de abertura ou outras autorizações para o café, pois o contexto não as menciona.
- Confirme se o seu caso se enquadra na ocupação de espaço público e na publicidade, e quais os anexos obrigatórios.
- Confirme os prazos de cada pedido, pois não estão indicados no contexto.

**Fontes:**
1. [Ocupação de espaço público](https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI09WUC1TTyNPVlAtU08=)
2. [Publicidade](https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI1BVQiNQVUI=)
3. [Autorização especial de horário de funcionamento](https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI0hPUi5FWEMjSE9SLkVYQw==)

*Informação gerada automaticamente a partir do portal da Câmara. Em caso de dúvida: Câmara Municipal de Guimarães · telefone 253 421 200 · geral@cm-guimaraes.pt.*

PDF gerado com sucesso: plano_1792748261.pdf


In [ ]:
def chat_fn(message, history):
    return formatar_resposta(
        rag_answer(message)
    )

print("✅ chat_fn criada")

✅ chat_fn criada


In [ ]:
# Agora o teste corre de forma 100% estável e rápida usando o modelo local
print(chat_fn(
    "Como peço o cartão municipal sénior?",
    []
))

Para pedir o Cartão Municipal Sénior, tem duas formas:

**Online:**
1. Registe-se no serviço online da Câmara (ou autentique-se, se já tiver conta).
2. Preencha o formulário online "Cartão Municipal Sénior".
3. Anexe os documentos necessários (ver abaixo).

**Presencial:**
1. Descarregue o requerimento "Cartão Municipal Sénior".
2. Preencha-o e entregue presencialmente com os documentos necessários.

**Requisitos:**
- Ter idade igual ou superior a 65 anos.
- Residir no concelho de Guimarães.

**Documentos a anexar (para segunda via):**
- Comprovativo de residência (certidão de eleitor ou atestado de residência da Junta de Freguesia) — em PDF.
- Fotografia atualizada — em JPG.

**Taxa (segunda via):** 1,00 €.

Se precisar de mais informações, contacte a Câmara Municipal de Guimarães: telefone 253 421 200 ou e-mail geral@cm-guimaraes.pt.

**Fontes:**
- [Cartão municipal sénior - online](https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI0NBL

In [ ]:
import time

inicio = time.time()

resultado = rag_answer(
    "Como peço o cartão municipal sénior?"
)

fim = time.time()

print("TEMPO:", round(fim - inicio, 2), "segundos")
print(resultado)

TEMPO: 0.0 segundos
{'answer': 'Para pedir o Cartão Municipal Sénior, tem duas formas:\n\n**Online:**\n1. Registe-se no serviço online da Câmara (ou autentique-se, se já tiver conta).\n2. Preencha o formulário online "Cartão Municipal Sénior".\n3. Anexe os documentos necessários (ver abaixo).\n\n**Presencial:**\n1. Descarregue o requerimento "Cartão Municipal Sénior".\n2. Preencha-o e entregue presencialmente com os documentos necessários.\n\n**Requisitos:**\n- Ter idade igual ou superior a 65 anos.\n- Residir no concelho de Guimarães.\n\n**Documentos a anexar (para segunda via):**\n- Comprovativo de residência (certidão de eleitor ou atestado de residência da Junta de Freguesia) — em PDF.\n- Fotografia atualizada — em JPG.\n\n**Taxa (segunda via):** 1,00 €.\n\nSe precisar de mais informações, contacte a Câmara Municipal de Guimarães: telefone 253 421 200 ou e-mail geral@cm-guimaraes.pt.', 'sources': [('Cartão municipal sénior - online', 'https://www.cm-guimaraes.pt/servicos-online/for

In [ ]:
if USAR_MODELO_LOCAL:
    def rag_answer_local(question):
        """Versão alternativa do RAG que utiliza o modelo SmolLM2 100% local e gratuito"""
        resultados = vectordb.similarity_search_with_relevance_scores(question, k=3)
        relevantes = [(doc, score) for doc, score in resultados if score >= MIN_SCORE]

        if not relevantes:
            return {"answer": NAO_SEI, "sources": []}

        context = "\n---\n".join(doc.page_content for doc, _ in relevantes)

        # Preparar as mensagens para o template local
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT_LOCAL},
            {"role": "user", "content": f"Contexto:\n{context}\n\nPergunta: {question}"}
        ]

        try:
            prompt_chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            outputs = local_generator(prompt_chat)
            answer = outputs[0]['generated_text'][len(prompt_chat):].strip()
        except Exception as e:
            answer = f"Ocorreu um erro ao gerar a resposta localmente: {e}"

        fontes, vistos = [], set()
        for doc, _ in relevantes:
            if doc.metadata["source"] not in vistos:
                vistos.add(doc.metadata["source"])
                fontes.append((doc.metadata["title"], doc.metadata["source"]))

        return {"answer": answer, "sources": fontes}

    # Testar a resposta local ultra-rápida
    inicio = time.time()
    resultado_local = rag_answer_local("Como peço o cartão municipal sénior?")
    fim = time.time()

    print("TEMPO LOCAL:", round(fim - inicio, 2), "segundos")
    print(formatar_resposta(resultado_local))

In [ ]:
# ==================================================
# Formulário: dados, validação e preenchimento automático (sem estado global)
# ==================================================
import os, re, json, time, html, tempfile
from urllib.parse import quote


def nif_valido(nif):
    """Valida o NIF português (9 dígitos e dígito de controlo)."""
    nif = re.sub(r"\D", "", str(nif or ""))
    if len(nif) != 9:
        return False
    soma = sum(int(d) * (9 - i) for i, d in enumerate(nif[:8]))
    controlo = 11 - soma % 11
    if controlo >= 10:
        controlo = 0
    return controlo == int(nif[8])


_RE_NUM = re.compile(r"(?<!\d)(\d{3}[ .-]?\d{3}[ .-]?\d{3})(?!\d)")


def extrair_numeros(texto):
    """Encontra NIF e telefone no texto, localmente (estes números NÃO são enviados ao modelo)."""
    texto = texto or ""
    nif = tel = ""
    fim_anterior = 0
    for m in _RE_NUM.finditer(texto):
        num = re.sub(r"\D", "", m.group(1))
        antes = texto[max(fim_anterior, m.start() - 25):m.start()].lower()   # só o texto entre o número anterior e este
        fim_anterior = m.end()
        if re.search(r"nif|contribuinte", antes):
            nif = nif or num
        elif re.search(r"telefone|telem[oó]vel|tlm|contacto|n[uú]mero", antes) or num[0] == "9":
            tel = tel or num
        elif nif_valido(num):
            nif = nif or num
        else:
            tel = tel or num
    return nif, tel


def mascarar_numeros(texto):
    return _RE_NUM.sub("[número]", texto or "")


_PISTAS_DADOS = re.compile(r"chamo|meu nome|sou (o|a) |moro|morada|resido|vivo (em|na|no)|localiza", re.I)

PROMPT_EXTRACAO = """Extrai da mensagem do utilizador, se existirem: o nome completo, a morada ou localização, e uma descrição curta (no máximo 20 palavras) do que a pessoa pretende.
Responde APENAS com um objeto JSON com as chaves "nome", "morada" e "descricao". Deixa vazio o que não existir. Não inventes nada.

Mensagem: __MSG__"""


def extrair_texto_com_llm(mensagem):
    """Pede ao modelo o nome, a morada e a descrição. A mensagem vai SEM números de NIF/telefone."""
    vazio = {"nome": "", "morada": "", "descricao": ""}
    if not _PISTAS_DADOS.search(mensagem or ""):
        return vazio
    try:
        resposta = chamar_llm(PROMPT_EXTRACAO.replace("__MSG__", mascarar_numeros(mensagem)), tentativas=1)
        dados = parse_json(resposta)
        return {k: str(dados.get(k) or "").strip() for k in vazio}
    except Exception as e:
        print(f"Aviso na extração automática: {type(e).__name__}")
        return vazio


def completar_form(mensagem, nome, nif, tel, morada, desc):
    """Preenche só os campos que estão vazios; nunca apaga o que a pessoa escreveu."""
    n, t = extrair_numeros(mensagem)
    ext = extrair_texto_com_llm(mensagem)
    return (nome or ext["nome"], nif or n, tel or t, morada or ext["morada"], desc or ext["descricao"])


def nomes_ficheiros(files):
    """O Gradio entrega caminhos (str) nas versões recentes e objetos com .name nas antigas."""
    return [os.path.basename(str(getattr(f, "name", f))) for f in (files or [])]


def painel_form(nome, nif, tel, morada, desc, files=None):
    """Painel espelho em HTML. O texto escrito pela pessoa é sempre 'escapado'."""
    def e(v):
        v = str(v or "").strip()
        return html.escape(v) if v else "<i style='color:#888'>Não preenchido</i>"

    nif_txt = e(nif)
    if str(nif or "").strip():
        nif_txt += " ✔" if nif_valido(nif) else " <span style='color:#b00020'>⚠ NIF inválido</span>"

    anexos_html = "".join(f"<li>📎 {html.escape(n)}</li>" for n in nomes_ficheiros(files)) \
        or "<li style='color:#888'>Nenhum ficheiro anexado</li>"

    linhas = [("Requerente", e(nome)), ("NIF", nif_txt), ("Contacto", e(tel)),
              ("Localização", e(morada)), ("Pedido", e(desc))]
    tabela = "".join(
        f"<tr style='border-bottom:1px solid #eee'><th style='padding:8px 0;width:30%;color:#005CA9;text-align:left'>{k}:</th>"
        f"<td style='padding:8px 0'>{v}</td></tr>" for k, v in linhas)

    return f"""<div style="border:2px solid #005CA9;border-radius:8px;padding:15px;background:#f8f9fa;color:#333;font-family:sans-serif">
<h3 style="color:#005CA9;margin-top:0">🖥️ Pedido pré-preenchido (em tempo real)</h3>
<p style="font-size:12px;color:#5f6368;margin-top:-5px">Protótipo: os campos preenchem-se pela conversa e podem ser editados. Nada é enviado.</p>
<table style="width:100%;border-collapse:collapse;font-size:13px">{tabela}</table>
<h4 style="color:#005CA9;margin-bottom:5px">📂 Ficheiros</h4>
<ul style="margin-top:5px;padding-left:20px;font-size:13px">{anexos_html}</ul></div>"""


print("✅ Funções do formulário prontas.")

✅ Funções do formulário prontas.


In [ ]:
# ==================================================
# Acessibilidade por áudio: ouvir as respostas (TTS) e falar em vez de escrever (STT)
# ==================================================
from gtts import gTTS


def limpar_para_audio(texto):
    """Tira links, markdown e a lista de fontes, para a voz ler só o que interessa."""
    texto = str(texto or "")
    texto = re.split(r"\n\s*\*\*Fontes", texto)[0]            # não ler a lista de fontes
    texto = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", texto)    # [texto](url) -> texto
    texto = re.sub(r"https?://\S+", "", texto)
    texto = re.sub(r"[*#_`>|]", "", texto)
    texto = re.sub(r"^\s*[-•]\s+", "", texto, flags=re.M)     # marcadores de lista
    return texto.strip()[:1800]


def texto_para_audio(texto):
    """Devolve o caminho de um .mp3 com o texto lido em português de Portugal (ou None se falhar)."""
    limpo = limpar_para_audio(texto)
    if not limpo:
        return None
    try:
        tts = gTTS(text=limpo, lang="pt", tld="pt")
        f = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
        f.close()
        tts.save(f.name)
        return f.name
    except Exception as e:
        print(f"Erro na síntese de áudio: {type(e).__name__}: {e}")
        return None


import subprocess
import numpy as np


def _decodificar_16k(caminho):
    """Lê qualquer áudio com o ffmpeg e devolve ondas mono a 16 kHz (o que o Whisper espera)."""
    r = subprocess.run(["ffmpeg", "-nostdin", "-v", "error", "-i", caminho,
                        "-f", "s16le", "-ac", "1", "-ar", "16000", "-"],
                       capture_output=True, check=True)
    return np.frombuffer(r.stdout, np.int16).astype(np.float32) / 32768.0


WHISPER_MODEL = "small"     # "base" é mais rápido mas percebe pior; "small" é um bom equilíbrio
_whisper = None


def transcrever_audio(caminho):
    """Transcreve de verdade o que foi gravado (faster-whisper, local e gratuito)."""
    global _whisper
    if not caminho:
        return ""
    if _whisper is None:
        from faster_whisper import WhisperModel
        print("A carregar o modelo de voz (a 1.ª vez demora cerca de 1 minuto)...")
        _whisper = WhisperModel(WHISPER_MODEL, device="cpu", compute_type="int8")
    # Decodificamos com o ffmpeg e passamos as ondas ao modelo (evita incompatibilidades da biblioteca "av" no Colab)
    ondas = _decodificar_16k(caminho)
    segmentos, _ = _whisper.transcribe(ondas, language="pt", beam_size=1)
    return " ".join(s.text.strip() for s in segmentos).strip()


def _texto_msg(content):
    """O conteúdo de uma mensagem do Chatbot pode vir como texto ou como lista de blocos."""
    if isinstance(content, str):
        return content
    if isinstance(content, (list, tuple)):
        return " ".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in content)
    return str(content or "")


def ouvir_ultima(historico):
    """Lê em voz alta a última resposta do assistente."""
    for m in reversed(historico or []):
        if isinstance(m, dict) and m.get("role") == "assistant":
            return texto_para_audio(_texto_msg(m.get("content")))
        if isinstance(m, (list, tuple)) and len(m) == 2:
            return texto_para_audio(_texto_msg(m[1]))
    return None


print("✅ Módulo de acessibilidade pronto.")

✅ Módulo de acessibilidade pronto.


In [ ]:
# ==================================================
# Documentos a anexar: lidos da página do procedimento (a secção "Anexos" do portal)
# ==================================================
import unicodedata, html


def _sa(t):
    t = unicodedata.normalize("NFKD", str(t).lower())
    return "".join(c for c in t if not unicodedata.combining(c)).strip()


_ROTULOS_IGNORAR = {"*", "pesquisar", "procurar anexos", "anexos", "taxas", "procurar taxas"}


def extrair_anexos(texto):
    """Lê da página os anexos pedidos: nome, obrigatório, nº máximo de ficheiros, assinatura digital, extensões."""
    linhas = [l.strip() for l in str(texto).splitlines() if l.strip()]
    marcas = [i for i, l in enumerate(linhas) if re.match(r"n\.?\s*[ºo°]\s*maximo de ficheiros", _sa(l))]
    anexos = []
    for n, i in enumerate(marcas):
        nome = ""
        for j in range(i - 1, max(i - 4, -1), -1):          # o nome está logo antes do rótulo
            if _sa(linhas[j]) not in _ROTULOS_IGNORAR:
                nome = linhas[j]
                break
        fim = marcas[n + 1] - 1 if n + 1 < len(marcas) else len(linhas)
        bloco = linhas[i:min(fim, i + 14)]

        def valor(rotulo):
            for k, l in enumerate(bloco):
                if _sa(l).startswith(rotulo) and k + 1 < len(bloco):
                    return bloco[k + 1]
            return ""

        try:
            maximo = int(float(linhas[i + 1])) if i + 1 < len(linhas) else None
        except ValueError:
            maximo = None
        if nome:
            anexos.append({"nome": nome, "max": maximo,
                           "assinatura": _sa(valor("necessita assinatura")) == "sim",
                           "obrigatorio": _sa(valor("obrigatorio")) != "nao",
                           "extensoes": valor("extensoes permitidas")})
    return anexos


def painel_anexos(fontes):
    """HTML com os documentos a anexar do 1.º procedimento (entre as fontes da resposta) que os tenha."""
    por_url = {c["url"]: c for c in catalog}
    for titulo, url in fontes or []:
        item = por_url.get(url)
        anexos = extrair_anexos(item["text"]) if item else []
        if anexos:
            itens = ""
            for a in anexos:
                detalhes = ["obrigatório" if a["obrigatorio"] else "opcional"]
                if a["max"]:
                    detalhes.append(f"até {a['max']} ficheiro(s)")
                if a["extensoes"]:
                    detalhes.append(html.escape(a["extensoes"]))
                if a["assinatura"]:
                    detalhes.append("assinatura digital")
                itens += (f"<li style='margin-bottom:6px'>📎 <b>{html.escape(a['nome'])}</b><br>"
                          f"<span style='color:#5f6368;font-size:12px'>{' · '.join(detalhes)}</span></li>")
            return (f"<div style='border:2px solid #005CA9;border-radius:8px;padding:15px;background:#f8f9fa;color:#333;font-family:sans-serif'>"
                    f"<h3 style='color:#005CA9;margin-top:0'>📎 Documentos a anexar</h3>"
                    f"<p style='font-size:13px;margin-top:-5px'>Para: <b>{html.escape(titulo)}</b> "
                    f"(<a href='{html.escape(url)}' target='_blank'>ver no portal</a>)</p>"
                    f"<ul style='padding-left:20px;font-size:13px'>{itens}</ul>"
                    f"<p style='font-size:11px;color:#888'>Informação lida do portal da Câmara. Confirme sempre no portal.</p></div>")
    return ""


PAINEL_ANEXOS_VAZIO = ("<div style='border:1px dashed #9aa0a6;border-radius:8px;padding:12px;color:#5f6368;font-family:sans-serif;font-size:13px'>"
                       "📎 Os documentos a anexar aparecem aqui quando perguntar por um procedimento.</div>")

print("✅ Leitura dos documentos a anexar pronta.")

✅ Leitura dos documentos a anexar pronta.


In [ ]:
# ==================================================
# Conversa: texto e voz (mesmo fluxo), com preenchimento automático do formulário
# ==================================================

def _turno(mensagem, historico, nome, nif, tel, morada, desc, files, anexos_ant="", falar=False, prefixo=""):
    """Responde a uma mensagem e devolve (histórico, áudio, painel, nome, nif, tel, morada, desc, anexos)."""
    historico = list(historico or [])
    mensagem = (mensagem or "").strip()
    anexos = anexos_ant
    if mensagem:
        nome, nif, tel, morada, desc = completar_form(mensagem, nome, nif, tel, morada, desc)
        try:
            r = rag_answer(mensagem)
            resposta = formatar_resposta(r)
            try:
                anexos = painel_anexos(r.get("sources", [])) or anexos_ant
            except Exception as e:
                print(f"Aviso nos anexos: {type(e).__name__}: {e}")
        except Exception as e:
            resposta = f"Ocorreu um erro ao obter resposta do assistente ({type(e).__name__})."
        historico.append({"role": "user", "content": prefixo + mensagem})
        historico.append({"role": "assistant", "content": resposta})
        audio = texto_para_audio(resposta) if falar else None
    else:
        audio = None
    return historico, audio, painel_form(nome, nif, tel, morada, desc, files), nome, nif, tel, morada, desc, anexos


def chat_texto_fn(mensagem, historico, nome, nif, tel, morada, desc, files, anexos_ant):
    h, a, p, n, i, t, m, d, anx = _turno(mensagem, historico, nome, nif, tel, morada, desc, files, anexos_ant)
    return "", h, a, p, n, i, t, m, d, anx     # o "" limpa a caixa de texto


def chat_voz_fn(audio_path, historico, nome, nif, tel, morada, desc, files, anexos_ant):
    historico = list(historico or [])
    painel = painel_form(nome, nif, tel, morada, desc, files)
    if not audio_path:
        historico.append({"role": "assistant", "content": "Grave primeiro a sua pergunta e depois carregue em enviar."})
        return historico, None, painel, nome, nif, tel, morada, desc, anexos_ant
    try:
        transcricao = transcrever_audio(audio_path)
    except Exception as e:
        historico.append({"role": "assistant", "content":
                          f"Não consegui usar o reconhecimento de voz ({type(e).__name__}). Pode escrever a pergunta."})
        return historico, None, painel, nome, nif, tel, morada, desc, anexos_ant
    if not transcricao:
        historico.append({"role": "assistant", "content":
                          "Não consegui perceber o áudio. Tente falar mais perto do microfone ou escreva a pergunta."})
        return historico, None, painel, nome, nif, tel, morada, desc, anexos_ant
    return _turno(transcricao, historico, nome, nif, tel, morada, desc, files, anexos_ant, falar=True, prefixo="🎤 ")


print("✅ Fluxo de conversa pronto.")

✅ Fluxo de conversa pronto.


In [ ]:
# ==================================================
# Pedido pré-preenchido: PDF + JSON (para integração) + e-mail para a Câmara
# Nada é enviado automaticamente: o envio real exigiria uma integração com os serviços da Câmara.
# ==================================================
CONTACTO_EMAIL = "geral@cm-guimaraes.pt"


def dados_do_pedido(nome, nif, tel, morada, desc, files):
    return {
        "Nome": (nome or "").strip(),
        "NIF": (nif or "").strip(),
        "Telefone": (tel or "").strip(),
        "Morada / localização": (morada or "").strip(),
        "Descrição do pedido": (desc or "").strip(),
        "Anexos": nomes_ficheiros(files),
    }


def gerar_pdf_pedido(dados, ficheiro):
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import cm
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
    from xml.sax.saxutils import escape

    est = getSampleStyleSheet()
    corpo = est["BodyText"]
    peq = ParagraphStyle("peq", parent=corpo, fontSize=8, textColor=colors.grey)
    e = lambda t: escape(str(t)) if str(t).strip() else "—"

    linhas = [[Paragraph(f"<b>{escape(k)}</b>", corpo), Paragraph(e(v), corpo)]
              for k, v in dados.items() if k != "Anexos"]
    tabela = Table(linhas, colWidths=[5 * cm, 12 * cm])
    tabela.setStyle(TableStyle([("GRID", (0, 0), (-1, -1), 0.4, colors.lightgrey),
                                ("VALIGN", (0, 0), (-1, -1), "TOP")]))
    anexos = dados.get("Anexos") or []

    el = [
        Paragraph("Pedido pré-preenchido", est["Title"]),
        Paragraph("Protótipo Hack4Guimarães 2035 · documento de apoio, <b>não submetido</b>", corpo),
        Paragraph(f"Gerado em {time.strftime('%d/%m/%Y às %H:%M')}", peq),
        Spacer(1, 12), tabela, Spacer(1, 10),
        Paragraph("Anexos a juntar", est["Heading3"]),
    ]
    if anexos:
        el += [Paragraph(f"• {escape(a)}", corpo) for a in anexos]
    else:
        el.append(Paragraph("Nenhum anexo indicado.", corpo))
    el += [Spacer(1, 24), Paragraph(
        "Documento gerado automaticamente por um protótipo. Não foi submetido, não está assinado e não substitui "
        "o formulário nos Serviços Online do Município. Para submeter, use os Serviços Online em cm-guimaraes.pt "
        f"ou entregue nos serviços da Câmara. Contacto: {CONTACTO}.", peq)]

    SimpleDocTemplate(ficheiro, pagesize=A4, leftMargin=2 * cm, rightMargin=2 * cm, topMargin=2 * cm,
                      bottomMargin=2 * cm, title="Pedido pré-preenchido").build(el)
    return ficheiro


def link_email(dados):
    """Link 'mailto:' que abre o e-mail da pessoa já escrito (o PDF tem de ser anexado por ela)."""
    corpo = "Bom dia,\n\n" + "\n".join(f"{k}: {v}" for k, v in dados.items() if k != "Anexos" and v)
    if dados.get("Anexos"):
        corpo += "\n\nAnexos a juntar: " + ", ".join(dados["Anexos"])
    corpo += "\n\n(Junte o PDF do pedido a este e-mail.)"
    assunto = "Pedido pré-preenchido (protótipo Hack4Guimarães)"
    return f"mailto:{CONTACTO_EMAIL}?subject={quote(assunto)}&body={quote(corpo[:1500])}"


def gerar_pedido(nome, nif, tel, morada, desc, files):
    """Valida, gera o PDF e o JSON e devolve (mensagem, caminho do PDF, caminho do JSON)."""
    dados = dados_do_pedido(nome, nif, tel, morada, desc, files)
    faltas = []
    if not dados["Nome"]:
        faltas.append("o nome")
    if not dados["NIF"]:
        faltas.append("o NIF")
    elif not nif_valido(dados["NIF"]):
        faltas.append("um NIF válido (9 dígitos)")
    if faltas:
        return "❌ Falta preencher " + " e ".join(faltas) + ".", None, None

    try:
        base = os.path.join(tempfile.gettempdir(), f"pedido_{int(time.time())}")
        pdf = gerar_pdf_pedido(dados, base + ".pdf")
        ficheiro_json = base + ".json"
        with open(ficheiro_json, "w", encoding="utf-8") as f:
            json.dump({"tipo": "pedido_pre_preenchido", "estado": "nao_submetido",
                       "gerado_em": time.strftime("%Y-%m-%dT%H:%M:%S"), "requerente": dados},
                      f, ensure_ascii=False, indent=2)
    except Exception as e:
        import traceback
        traceback.print_exc()          # o detalhe aparece também na célula do Colab
        return f"❌ Não consegui gerar o PDF ({type(e).__name__}: {str(e)[:200]})", None, None

    msg = ("### ✅ Pedido gerado (nada foi enviado)\n"
           "Descarregue o PDF e o ficheiro de dados abaixo.\n\n"
           f"- [✉️ Abrir e-mail para a Câmara]({link_email(dados)}), depois junte o PDF.\n"
           "- Ou submeta nos **Serviços Online** do Município / entregue presencialmente.\n"
           "- O ficheiro JSON tem os mesmos dados em formato legível por sistemas, para uma futura integração.")
    return msg, pdf, ficheiro_json


print("✅ Geração do pedido pronta.")

✅ Geração do pedido pronta.


In [ ]:
# ==================================================
# Interface única (um só launch no fim)
# ==================================================
import gradio as gr

TITULO = "M.E.L — protótipo Hack4Guimarães"

GUIMARAES_CSS = """
body { background-color: #f4f6f9 !important; font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif !important; }
.gradio-container { max-width: 1350px !important; margin: auto !important; background-color: #ffffff !important; }
button.primary, .primary-btn {
    background: linear-gradient(135deg, #005CA9 0%, #003F7A 100%) !important;
    color: white !important; border: none !important; border-radius: 6px !important; font-weight: bold !important;
}
button.primary:hover { background: linear-gradient(135deg, #003F7A 0%, #00254D 100%) !important; }
"""

CABECALHO = """
<div style="display:flex;align-items:center;gap:15px;border-bottom:3px solid #005CA9;padding-bottom:15px;margin-bottom:15px;font-family:sans-serif">
  <div style="background:#005CA9;color:white;padding:10px 18px;font-weight:900;font-size:22px;border-radius:8px">M.E.L</div>
  <div>
    <h1 style="margin:0;color:#333;font-size:21px">Falar com a Câmara sem burocracia</h1>
    <p style="margin:0;color:#666;font-size:13px">Assistente acessível (texto e voz) · Protótipo Hack4Guimarães 2035 ·
    <b>não é um serviço oficial</b> da Câmara Municipal de Guimarães</p>
  </div>
</div>
"""

AVISO_DADOS = ("⚠️ **Demonstração: use dados fictícios.** O NIF e o telefone são reconhecidos localmente e **não** são enviados ao modelo; "
               "o nome, a morada e a descrição são analisados pelo modelo (DeepSeek) para preencher os campos. Nada é guardado nem enviado à Câmara.")



def criar_chatbot(**kw):
    """Compatível com várias versões do Gradio (formato de mensagens)."""
    try:
        return gr.Chatbot(type="messages", **kw)
    except TypeError:
        return gr.Chatbot(**kw)


try:
    demo.close()          # se esta célula for corrida outra vez, fecha a versão anterior
except Exception:
    pass

try:
    demo = gr.Blocks(title=TITULO, css=GUIMARAES_CSS)
except TypeError:         # versões em que o css passa para o launch()
    demo = gr.Blocks(title=TITULO)

with demo:
    gr.HTML(CABECALHO)

    with gr.Tab("💬 Assistente e formulário"):
        with gr.Row():
            with gr.Column(scale=6):
                gr.Markdown("### 💬 1. Escreva ou fale com o assistente")
                chatbot = criar_chatbot(label="Conversa", height=380)
                msg_input = gr.Textbox(label="A sua mensagem", placeholder="Escreva aqui e carregue em Enter...")
                with gr.Row():
                    btn_enviar = gr.Button("Enviar", variant="primary")
                    btn_ouvir = gr.Button("🔊 Ouvir a última resposta")
                gr.Markdown("#### 🎤 Prefere falar?")
                input_audio = gr.Audio(label="Grave a sua pergunta", type="filepath", sources=["microphone"])
                btn_voz = gr.Button("🎤 Enviar mensagem de voz", variant="primary")
                output_audio = gr.Audio(label="Resposta em áudio", autoplay=True)

            with gr.Column(scale=6):
                gr.Markdown("### 📋 2. Pedido pré-preenchido")
                gr.Markdown(AVISO_DADOS)
                espelho = gr.HTML(value=painel_form("", "", "", "", "", None))
                anexos_html = gr.HTML(value=PAINEL_ANEXOS_VAZIO)
                in_nome = gr.Textbox(label="Nome completo", placeholder="Preenche-se pela conversa, ou escreva aqui")
                in_nif = gr.Textbox(label="NIF")
                in_tel = gr.Textbox(label="Telefone")
                in_morada = gr.Textbox(label="Morada / localização")
                in_desc = gr.TextArea(label="Descrição do pedido")
                in_files = gr.File(label="Documentos a anexar (PDF, fotos)", file_count="multiple")
                btn_pdf = gr.Button("📄 Gerar pedido em PDF", variant="primary")
                out_msg = gr.Markdown()
                out_pdf = gr.File(label="PDF do pedido")
                out_json = gr.File(label="Dados em JSON (para integração)")

    with gr.Tab("📋 Planear cenário complexo"):
        gr.Markdown("### 📋 Planeamento de atividades e obras\n"
                    "Descreva a situação e o assistente prepara a lista de procedimentos, documentos, custos e prazos "
                    "**indicados no portal** (o que o portal não diz aparece como «Não indicado»).")
        situacao_in = gr.Textbox(label="A minha situação", lines=2, placeholder="Ex.: vou abrir um café no centro de Guimarães")
        btn_plano = gr.Button("Criar o meu plano", variant="primary")
        plano_out = gr.Markdown()
        with gr.Row():
            pdf_out = gr.File(label="Checklist em PDF")
            audio_plano = gr.Audio(label="Ouvir o plano", autoplay=False)
        btn_ouvir_plano = gr.Button("🔊 Ouvir o plano")




    # ---------------- ligações
    campos = [in_nome, in_nif, in_tel, in_morada, in_desc, in_files]
    entradas_chat = [msg_input, chatbot] + campos + [anexos_html]
    saidas_texto = [msg_input, chatbot, output_audio, espelho, in_nome, in_nif, in_tel, in_morada, in_desc, anexos_html]
    saidas_voz = [chatbot, output_audio, espelho, in_nome, in_nif, in_tel, in_morada, in_desc, anexos_html]

    msg_input.submit(chat_texto_fn, inputs=entradas_chat, outputs=saidas_texto)
    btn_enviar.click(chat_texto_fn, inputs=entradas_chat, outputs=saidas_texto)
    btn_voz.click(chat_voz_fn, inputs=[input_audio, chatbot] + campos + [anexos_html], outputs=saidas_voz)
    btn_ouvir.click(ouvir_ultima, inputs=chatbot, outputs=output_audio)

    for campo in campos:
        campo.change(painel_form, inputs=campos, outputs=espelho)

    btn_pdf.click(gerar_pedido, inputs=campos, outputs=[out_msg, out_pdf, out_json])

    btn_plano.click(planear, inputs=situacao_in, outputs=[plano_out, pdf_out])
    btn_ouvir_plano.click(texto_para_audio, inputs=plano_out, outputs=audio_plano)

try:
    demo.launch(share=True, css=GUIMARAES_CSS)
except TypeError:         # versões em que o css já foi dado ao gr.Blocks()
    demo.launch(share=True)

/tmp/ipykernel_4852/3416043963.py:48: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  demo = gr.Blocks(title=TITULO, css=GUIMARAES_CSS)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://42d3ff56705f88b8f9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
